[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dennymarcels/dubbsystem/blob/main/notebooks/colab_dubb_demo.ipynb)

# DubbSystem Colab Demo
This notebook installs DubbSystem from git, uploads an MP4, runs the dubbing pipeline, and previews the dubbed result.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Before You Run
Update the repository URL in the next cell before executing the notebook. A GPU runtime is strongly recommended for transcription, large-model translation, and voice cloning.
If you have reviewed and accepted the applicable Coqui XTTS terms, set `COQUI_TOS_AGREED = "1"` in the configuration cell to skip the interactive license prompt during model download.
The notebook defaults to American English dubbing with `TARGET_LANGUAGE = "en-us"` and a larger NLLB translation model.

In [ ]:
import sys
REPO_URL = "https://github.com/dennymarcels/dubbsystem.git"
TARGET_LANGUAGE = "en-us"
TRANSCRIPTION_MODEL = "large-v3"
TRANSLATION_MODEL = "facebook/nllb-200-3.3B"
VOICE_SAMPLE_SECONDS = 60
LOG_LEVEL = "INFO"
COQUI_TOS_AGREED = "1"  # Set to "1" only after you review and accept the applicable Coqui terms.
PYTHON_BIN = "python3.11" if sys.version_info >= (3, 12) else "python3"
print(f"Notebook runtime: {sys.version.split()[0]}")
print(f"Installer Python: {PYTHON_BIN}")
print(f"Target language: {TARGET_LANGUAGE}")
print(f"Transcription model: {TRANSCRIPTION_MODEL}")
print(f"Translation model: {TRANSLATION_MODEL}")
print(f"Voice sample seconds: {VOICE_SAMPLE_SECONDS}")

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg python3.11 python3.11-venv
!git clone "$REPO_URL" /content/DubbSystem
%cd /content/DubbSystem
!rm -rf .venv
!$PYTHON_BIN -m venv .venv
!./.venv/bin/python -m pip install --upgrade pip "setuptools<81" wheel
!./.venv/bin/python -m pip install -e .
!./.venv/bin/python -m pip install "setuptools<81"

## Upload an MP4
Use the file picker to upload one source MP4 video to the Colab runtime.

In [ ]:
from google.colab import files

In [ ]:
input_path = "/content/drive/MyDrive/Cursos IA Expert/MLOps/vídeos/1.1 WSL.mp4"
print(f"Uploaded: {input_path}")

In [ ]:
from pathlib import Path
input_file = Path(input_path)
output_file = input_file.with_name(f"{input_file.stem}_dubbed{input_file.suffix}")
print(f"Expected output: {output_file}")

## Whole Process Reference
Keep this cell commented out. It shows the packaged end-to-end CLI invocation, but the staged workflow below is the recommended path when you want to inspect files after each phase.

In [ ]:
# Uncomment this command only if you want to run the packaged CLI end to end.
# The step-by-step workflow below writes the same intermediate artifacts in a way that is easier to inspect.
# !COQUI_TOS_AGREED={COQUI_TOS_AGREED} ./.venv/bin/dubb "{input_path}" "{output_file}" --target-language "{TARGET_LANGUAGE}" --transcription-model "{TRANSCRIPTION_MODEL}" --translation-model "{TRANSLATION_MODEL}" --voice-sample-seconds {VOICE_SAMPLE_SECONDS} --log-level "{LOG_LEVEL}" --keep-temp

## Step-by-Step Workflow
Run the cells below in order. Each phase writes files into the working directory so you can inspect audio, JSON artifacts, and aligned synthesis outputs before continuing.

In [ ]:
import json
import os
import subprocess
from pathlib import Path
from IPython.display import Audio, Markdown, Video, display

repo_root = Path("/content/DubbSystem")
venv_dubb = repo_root / ".venv" / "bin" / "dubb"
work_dir = input_file.parent / ".dubb_tmp" / input_file.stem
source_audio = work_dir / "source.wav"
speaker_sample = work_dir / "speaker_sample.wav"
speaker_sample_selection_path = work_dir / "speaker_sample.selection.json"
transcript_source_path = work_dir / "transcript.source.json"
transcript_translated_path = work_dir / "transcript.translated.json"
chunks_path = work_dir / "synthesis_chunks.json"
manifest_path = work_dir / "synthesis_manifest.json"
dubbed_audio = work_dir / "dubbed_track.wav"


def dubb_step_command(step_name: str) -> list[str]:
    return [
        str(venv_dubb),
        step_name,
        str(input_file),
        str(output_file),
        "--target-language",
        TARGET_LANGUAGE,
        "--transcription-model",
        TRANSCRIPTION_MODEL,
        "--translation-model",
        TRANSLATION_MODEL,
        "--voice-sample-seconds",
        str(VOICE_SAMPLE_SECONDS),
        "--log-level",
        LOG_LEVEL,
    ]


def run_step(step_name: str) -> None:
    command = dubb_step_command(step_name)
    print("Running:", " ".join(command))
    environment = os.environ.copy()
    environment["COQUI_TOS_AGREED"] = COQUI_TOS_AGREED
    subprocess.run(command, check=True, cwd=str(repo_root), env=environment)


print(f"Artifacts will be written to: {work_dir}")

## Step 1: Extract Audio
This extracts the source audio track into a WAV file for inspection.

In [ ]:
run_step("extract-audio")
print(f"Extracted audio: {source_audio}")
display(Markdown("### Source Audio"))
display(Audio(str(source_audio)))

## Step 2: Transcribe
This generates the raw transcript JSON with timestamps and detected source language. The speaker sample step uses this transcript to choose the best 60 seconds.

In [ ]:
run_step("transcribe")
transcript_source_payload = json.loads(transcript_source_path.read_text(encoding="utf-8"))
print(f"Raw transcript artifact: {transcript_source_path}")
print(f"Detected source language: {transcript_source_payload.get('source_language', 'auto')}")
display(Markdown("### Raw Transcript Sample"))
print(json.dumps(transcript_source_payload["segments"][:3], indent=2, ensure_ascii=False))

## Step 3: Create Speaker Sample
This creates a cleaned 60-second speaker reference WAV from the best transcript-ranked speech regions and saves the selection metadata for inspection.

In [ ]:
run_step("create-speaker-sample")
speaker_sample_selection_payload = json.loads(speaker_sample_selection_path.read_text(encoding="utf-8"))
print(f"Speaker sample: {speaker_sample}")
print(f"Speaker sample selection artifact: {speaker_sample_selection_path}")
display(Markdown("### Speaker Sample"))
display(Audio(str(speaker_sample)))
display(Markdown("### Speaker Sample Selection"))
print(json.dumps(speaker_sample_selection_payload["selected_segments"][:3], indent=2, ensure_ascii=False))

## Step 4: Translate
This generates the translated transcript JSON in the target language.

In [ ]:
run_step("translate")
transcript_translated_payload = json.loads(transcript_translated_path.read_text(encoding="utf-8"))
print(f"Translated transcript artifact: {transcript_translated_path}")
display(Markdown("### Translated Transcript Sample"))
print(json.dumps(transcript_translated_payload["segments"][:3], indent=2, ensure_ascii=False))

## Step 5: Prepare Synthesis Chunks
This merges translated segments into chunk windows that will be synthesized and time-aligned.

In [ ]:
run_step("prepare-synthesis-chunks")
chunks_payload = json.loads(chunks_path.read_text(encoding="utf-8"))
print(f"Synthesis chunks artifact: {chunks_path}")
display(Markdown("### Synthesis Chunk Sample"))
print(json.dumps(chunks_payload[:3], indent=2, ensure_ascii=False))

## Step 6: Synthesize Segments
This generates aligned segment WAV files and the synthesis manifest for inspection.

In [ ]:
run_step("synthesize")
manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
print(f"Synthesis manifest: {manifest_path}")
display(Markdown("### Synthesis Manifest Sample"))
print(json.dumps(manifest_payload[:3], indent=2, ensure_ascii=False))
if manifest_payload:
    display(Markdown("### First Aligned Segment"))
    display(Audio(manifest_payload[0]["aligned_audio"]))

## Step 7: Compose Dubbed Audio
This overlays the aligned synthesized segments into the final dubbed WAV track.

In [ ]:
run_step("compose-audio")
print(f"Dubbed audio: {dubbed_audio}")
display(Markdown("### Dubbed Audio"))
display(Audio(str(dubbed_audio)))

## Step 8: Mux Video
This writes the final dubbed MP4 and previews it inline.

In [ ]:
run_step("mux-video")
print(f"Dubbed video: {output_file}")
display(Markdown("### Dubbed Video"))
display(Video(str(output_file), embed=True))

## Inspect Generated Files
This lists every artifact written by the staged workflow.

In [ ]:
artifact_files = sorted(
    path.relative_to(work_dir).as_posix()
    for path in work_dir.rglob("*")
    if path.is_file()
)
for artifact_file in artifact_files:
    print(artifact_file)